In [0]:
# ============================================================
# Load each bronze table into its own named Spark DataFrame
# ============================================================

# Location-related tables
locations_main = spark.table("domiciliarycare.bronze.cqc_kent_locations_main")
regulated_activities = spark.table("domiciliarycare.bronze.cqc_kent_regulated_activities")
gac_service_types = spark.table("domiciliarycare.bronze.cqc_kent_gac_service_types")
specialisms = spark.table("domiciliarycare.bronze.cqc_kent_specialisms")
inspection_categories = spark.table("domiciliarycare.bronze.cqc_kent_inspection_categories")
inspection_areas = spark.table("domiciliarycare.bronze.cqc_kent_inspection_areas")
reports = spark.table("domiciliarycare.bronze.cqc_kent_reports")
ratings = spark.table("domiciliarycare.bronze.cqc_kent_ratings")

# Provider-related tables
providers_main = spark.table("domiciliarycare.bronze.cqc_kent_providers_main")
provider_locations_bridge = spark.table("domiciliarycare.bronze.cqc_kent_provider_locations_bridge")
provider_contacts = spark.table("domiciliarycare.bronze.cqc_kent_provider_contacts")
provider_relationships = spark.table("domiciliarycare.bronze.cqc_kent_provider_relationships")
provider_regulated_activities = spark.table("domiciliarycare.bronze.cqc_kent_provider_regulated_activities")
provider_inspection_categories = spark.table("domiciliarycare.bronze.cqc_kent_provider_inspection_categories")
provider_inspection_areas = spark.table("domiciliarycare.bronze.cqc_kent_provider_inspection_areas")
provider_reports = spark.table("domiciliarycare.bronze.cqc_kent_provider_reports")
provider_ratings = spark.table("domiciliarycare.bronze.cqc_kent_provider_ratings")

print("All tables loaded.")

All tables loaded.


In [0]:
locations_main.show(6)
locations_main.count()
locations_main = locations_main.drop('number_of_beds','organisation_type')
# locations_main.select('postal_code').distinct().display()

+-------------+------------+-----------------+---------------+--------------------+--------------+--------------------+--------------+--------------------+--------+-------------------+-----------------+--------+----------+---------+---------+----------------------+--------------------+--------------------+------------------------+----------+-----------+------------+-----------------+--------------------+--------------+--------------------+---------------+--------------------+----------------------------+
|  location_id| provider_id|organisation_type|           type|                name|onspd_ccg_code|      onspd_ccg_name|onspd_icb_code|      onspd_icb_name|ods_code|registration_status|registration_date|dormancy|  latitude|longitude|care_home|inspection_directorate|postal_address_line1|postal_address_line2|postal_address_town_city|    region|postal_code|        uprn|main_phone_number|             website|number_of_beds|        constituency|local_authority|last_inspection_date|last_repor

In [0]:
import re

def validate_uk_postcode(postcode):
    if not postcode or str(postcode).strip() in ("", "nan", "None"):
        return False, "missing"
    pc = str(postcode).strip().upper()
    pattern = r"^[A-Z]{1,2}\d[A-Z\d]?\s\d[A-Z]{2}$"
    if re.match(pattern, pc):
        return True, None
    return False, "does not match UK postcode structure"

results = locations_main.toPandas()["postal_code"].apply(validate_uk_postcode)
invalid_count = sum(1 for valid, _ in results if not valid)
print(f"Invalid postcodes found: {invalid_count} / {len(results)}")

Invalid postcodes found: 0 / 458


In [0]:
from pyspark.sql.functions import col
regulated_activities.show(6)
regulated_activities.select('activity_name').distinct().display()
regulated_activities = regulated_activities.filter(col('activity_name') == "Personal care")
regulated_activities.select('activity_name').distinct().display()


+-------------+-------------+-------------+-------------+------------------+-------------------+------------------+
|  location_id|activity_name|activity_code|contact_title|contact_given_name|contact_family_name|     contact_roles|
+-------------+-------------+-------------+-------------+------------------+-------------------+------------------+
|1-10064258273|Personal care|          RA1|         Miss|              Anna|            Fleming|Registered Manager|
|1-10087632956|Personal care|          RA1|           Mr|             David|               Babu|Registered Manager|
|1-10200463066|Personal care|          RA1|         None|              None|               None|              None|
|1-10204301199|Personal care|          RA1|          Mrs|         Karen Eva|             Gowers|Registered Manager|
|1-10212079624|Personal care|          RA1|           Mr|        Luke Bryan|       Ilett-Mackie|Registered Manager|
|1-10212079624|Personal care|          RA1|           Ms|      Rebecca A

activity_name
Personal care
"Treatment of disease, disorder or injury"
Accommodation for persons who require nursing or personal care
Diagnostic and screening procedures


activity_name
Personal care


In [0]:
gac_service_types.show(6)
gac_service_types.select('service_type_description','service_type_name').distinct().display()
gac_service_types = gac_service_types.filter(col('service_type_name').isin('Homecare agencies','Supported living','shared_lives','Supported housing'))

+-------------+-----------------+------------------------+
|  location_id|service_type_name|service_type_description|
+-------------+-----------------+------------------------+
|1-10064258273|Homecare agencies|    Domiciliary care ...|
|1-10087632956|Homecare agencies|    Domiciliary care ...|
|1-10200463066|Homecare agencies|    Domiciliary care ...|
|1-10204301199|Homecare agencies|    Domiciliary care ...|
|1-10204301199| Supported living|    Supported living ...|
|1-10212079624|Homecare agencies|    Domiciliary care ...|
+-------------+-----------------+------------------------+
only showing top 6 rows


service_type_description,service_type_name
Domiciliary care service,Homecare agencies
Supported living service,Supported living
Care home service without nursing,Residential homes
Doctors consultation service,Doctors/Gps
Community health care services - Nurses Agency only,Community services - Nursing
Rehabilitation services,Rehabilitation (illness/injury)
Extra Care housing services,Supported housing
Shared Lives,Shared lives
Care home service with nursing,Nursing homes
Community based services for people with a learning disability,Community services - Learning disabilities


In [0]:
specialisms.select('specialism_name').distinct().display()

specialism_name
Learning disabilities
Caring for adults under 65 yrs
Caring for adults over 65 yrs
Substance misuse problems
Physical disabilities
Dementia
Sensory impairment
Services for everyone
Caring for children
Mental health conditions


In [0]:
reports.show(6)

+-------------+--------------------+-----------+----------------+--------------------+-----------+
|  location_id|      report_link_id|report_date|first_visit_date|          report_uri|report_type|
+-------------+--------------------+-----------+----------------+--------------------+-----------+
|1-10064258273|a2d37301-7573-41f...| 2022-03-18|      2022-02-28|/reports/a2d37301...|   Location|
|1-10087632956|49e05bbb-75f3-4ab...| 2022-05-25|      2022-04-13|/reports/49e05bbb...|   Location|
|1-10200463066|4f4c83c7-7cd1-4dd...| 2022-12-30|      2022-08-31|/reports/4f4c83c7...|   Location|
|1-10204301199|f81efee2-771a-4dd...| 2022-11-11|      2022-06-08|/reports/f81efee2...|   Location|
|1-10204301199|c988726f-533c-4f8...| 2022-03-23|      2022-02-01|/reports/c988726f...|   Location|
|1-10212079624|aff6a826-a56e-426...| 2022-07-01|      2022-05-11|/reports/aff6a826...|   Location|
+-------------+--------------------+-----------+----------------+--------------------+-----------+
only showi

In [0]:
ratings.show(6)

+-------------+-------------+------+-----------+--------------------+
|  location_id|question_name|rating|report_date|      report_link_id|
+-------------+-------------+------+-----------+--------------------+
|1-10064258273|         Safe|  Good| 2022-03-18|a2d37301-7573-41f...|
|1-10064258273|     Well-led|  Good| 2022-03-18|a2d37301-7573-41f...|
|1-10064258273|       Caring|  Good| 2022-03-18|a2d37301-7573-41f...|
|1-10064258273|   Responsive|  Good| 2022-03-18|a2d37301-7573-41f...|
|1-10064258273|    Effective|  Good| 2022-03-18|a2d37301-7573-41f...|
|1-10204301199|         Safe|  Good| 2022-11-11|f81efee2-771a-4dd...|
+-------------+-------------+------+-----------+--------------------+
only showing top 6 rows


In [0]:
providers_main.show(6)


+-------------+--------------+---------------+--------------------+--------+----------+--------+-------------------+-----------------+----------------------+--------------+--------------------+--------------------+--------------------+------------------------+---------------------+----------+-----------+-------------+-------------------+------------+----------+----------+--------------+--------------------+-----------------+----------------------+--------------------+---------------+--------------------+----------------------------+
|  provider_id|ownership_type|           type|                name|brand_id|brand_name|ods_code|registration_status|registration_date|companies_house_number|charity_number|             website|postal_address_line1|postal_address_line2|postal_address_town_city|postal_address_county|    region|postal_code|also_known_as|deregistration_date|        uprn|  latitude| longitude|onspd_icb_code|      onspd_icb_name|main_phone_number|inspection_directorate|        co

In [0]:
provider_locations_bridge.show(6)


+-----------+------------+
|provider_id| location_id|
+-----------+------------+
|1-102643104|1-4049603495|
|1-102643104|1-4158241430|
|1-102643104|1-4158241497|
|1-102643104|1-4180628384|
|1-102643104| 1-420211675|
|1-102643104| 1-466143187|
+-----------+------------+
only showing top 6 rows


In [0]:
results = providers_main.toPandas()["postal_code"].apply(validate_uk_postcode)
invalid_count = sum(1 for valid, _ in results if not valid)
print(f"Invalid postcodes found: {invalid_count} / {len(results)}")

Invalid postcodes found: 0 / 401


In [0]:
provider_contacts.show(6)

+-----------+-------------+--------------------+-------------------+-------------+
|provider_id|contact_title|  contact_given_name|contact_family_name|contact_roles|
+-----------+-------------+--------------------+-------------------+-------------+
|1-101625791|           Mr|Benjamin Alexande...|             Mirsky|      Partner|
|1-101625791|           Mr|               David|             Mirsky|      Partner|
|1-101625791|          Mrs|          Jacqueline|             Mirsky|      Partner|
|1-101703352|           Mr|                Gary|              White|      Partner|
|1-101703352|          Mrs|                June|              White|      Partner|
|1-101720227|          Mrs|             Marilyn|             Squire|      Partner|
+-----------+-------------+--------------------+-------------------+-------------+
only showing top 6 rows


In [0]:
provider_relationships.show(6)


+------------+-------------------+---------------------+-----------------+-------------------+
| provider_id|related_provider_id|related_provider_name|relationship_type|             reason|
+------------+-------------------+---------------------+-----------------+-------------------+
| 1-101641035|        1-101639703| Care In The Home Ltd| HSCA Predecessor|       New Provider|
| 1-101641035|        1-101724782|    Complete Homecare| HSCA Predecessor|Legal Entity Change|
| 1-101696436|        1-131467205| Nelson Park Care ...| HSCA Predecessor|Legal Entity Change|
|1-1024772695|        1-185350867| Mr Simon Rowe and...| HSCA Predecessor|Legal Entity Change|
| 1-102642564|        1-118165881| Kathryn Homes Lim...| HSCA Predecessor|       New Provider|
| 1-102642564|                5NY| Bradford and Aire...| HSCA Predecessor|       NHS Transfer|
+------------+-------------------+---------------------+-----------------+-------------------+
only showing top 6 rows


In [0]:
provider_regulated_activities.show(6)


+-------------+--------------------+-------------+-------------+--------------------+-------------------+
|  provider_id|       activity_name|activity_code|nominee_title|  nominee_given_name|nominee_family_name|
+-------------+--------------------+-------------+-------------+--------------------+-------------------+
|1-10006742735|       Personal care|          RA1|           Mr|       Jubbin Vinish|              Jacob|
|1-10058658365|       Personal care|          RA1|         Miss|Sandra Dorothy Ma...|         Nyakupinda|
|  1-101614760|       Personal care|          RA1|           Mr|        Mark Douglas|              Allen|
|  1-101614760|Accommodation for...|          RA2|           Mr|        Mark Douglas|              Allen|
|  1-101615228|       Personal care|          RA1|           Ms|    Lianne Elizabeth|            Rollins|
|  1-101615228|Accommodation for...|          RA2|           Ms|    Lianne Elizabeth|            Rollins|
+-------------+--------------------+----------

In [0]:
provider_inspection_categories.show(6)


+-------------+-------------+----------+--------------------+
|  provider_id|category_code|is_primary|       category_name|
+-------------+-------------+----------+--------------------+
|1-10006742735|           S2|      true|Community based a...|
|1-10058658365|           S2|      true|Community based a...|
|  1-101614760|           S1|      true|Residential socia...|
|  1-101614760|           S2|      None|Community based a...|
|  1-101615228|           S1|      true|Residential socia...|
|  1-101615228|           S2|      None|Community based a...|
+-------------+-------------+----------+--------------------+
only showing top 6 rows
